<a href="https://colab.research.google.com/github/khanhtran94/crawTTV/blob/master/predicete_vietlot_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [63]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

# =============================================================
# 1. TẢI DỮ LIỆU THẬT POWER 6/55 TỪ GITHUB
# =============================================================
url = "https://raw.githubusercontent.com/vietvudanh/vietlott-data/main/data/power655.jsonl"

print("Đang tải dữ liệu Power 6/55 từ GitHub...")
df = pd.read_json(url, lines=True)

# Lấy 6 số chính trong mỗi kỳ quay (bỏ qua số đặc biệt thứ 7)
data = [row[:6] for row in df['result']]

num_draws = len(data)
total_numbers = 55  # Power 6/55 gồm các số từ 1 đến 55

print(f"\n=== ĐÃ TẢI THÀNH CÔNG {num_draws} KỲ QUAY ===")
print("5 kỳ gần nhất từ dữ liệu thật:")
for i, d in enumerate(data[-5:], start=num_draws - 4):
    print(f"Kỳ {i}: {sorted(d)}")

# Hàm hỗ trợ tạo 3 vé dựa trên phân phối xác suất
def sample_ticket_from_probs(probabilities, num_tickets=3, temperature=1.0):
    probs = np.clip(probabilities, 1e-8, None)
    scaled_probs = np.exp(np.log(probs) / temperature)
    norm_probs = scaled_probs / np.sum(scaled_probs)

    tickets = []
    for _ in range(num_tickets):
        chosen = np.random.choice(range(1, total_numbers + 1), size=6, replace=False, p=norm_probs)
        tickets.append(sorted(list(chosen)))
    return tickets


# =============================================================
# 2. STRATEGY 2: LSTM (PYTORCH)
# =============================================================
class LSTMModel(nn.Module):
    def __init__(self, input_size=55, hidden_size=64):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

seq_len = 5
X_seq, y_seq = [], []
for i in range(seq_len, len(data)):              # Iterate up to the second to last element
    seq_in = [np.eye(total_numbers)[np.array(draw) - 1].sum(axis=0) for draw in data[i - seq_len:i]]
    seq_out = np.eye(total_numbers)[np.array(data[i]) - 1].sum(axis=0)
    X_seq.append(seq_in)
    y_seq.append(seq_out)

X_seq_t = torch.tensor(np.array(X_seq), dtype=torch.float32)
y_seq_t = torch.tensor(np.array(y_seq), dtype=torch.float32)

lstm = LSTMModel(input_size=total_numbers)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(lstm.parameters(), lr=0.008)

print("\nĐang huấn luyện LSTM...")
lstm.train()
for epoch in range(60):
    optimizer.zero_grad()
    outputs = lstm(X_seq_t)
    loss = criterion(outputs, y_seq_t)
    loss.backward()
    optimizer.step()

last_seq = [np.eye(total_numbers)[np.array(draw) - 1].sum(axis=0) for draw in data[-seq_len:]]
last_seq_t = torch.tensor(np.array([last_seq]), dtype=torch.float32)

lstm.eval()
with torch.no_grad():
    logits = lstm(last_seq_t)
    lstm_probs = torch.sigmoid(logits).numpy()[0]

print("\n" + "="*40)
print("  STRATEGY 2: LSTM - 3 TICKETS (Temperature 1.5)")
print("="*40)

# Chỉ sử dụng Temperature 1.5
target_temp = 0.7
final_lstm_tickets = [] # Biến mới để lưu trữ các vé
for idx in range(1, 4): # Tạo 3 vé với Temperature 1.5
    ticket = sample_ticket_from_probs(lstm_probs, num_tickets=1, temperature=target_temp)[0]
    final_lstm_tickets.append({
        'id': idx,
        'temp': target_temp,
        'ticket': ticket
    })
    print(f"Ticket {idx} (Temp {target_temp:.1f}): {ticket}")


Đang tải dữ liệu Power 6/55 từ GitHub...

=== ĐÃ TẢI THÀNH CÔNG 1402 KỲ QUAY ===
5 kỳ gần nhất từ dữ liệu thật:
Kỳ 1398: [24, 27, 32, 36, 38, 47]
Kỳ 1399: [6, 11, 25, 27, 37, 45]
Kỳ 1400: [4, 7, 11, 18, 22, 25]
Kỳ 1401: [1, 3, 9, 11, 41, 46]
Kỳ 1402: [1, 13, 23, 25, 26, 28]

Đang huấn luyện LSTM...

  STRATEGY 2: LSTM - 3 TICKETS (Temperature 1.5)
Ticket 1 (Temp 0.7): [np.int64(16), np.int64(19), np.int64(22), np.int64(24), np.int64(43), np.int64(45)]
Ticket 2 (Temp 0.7): [np.int64(3), np.int64(7), np.int64(17), np.int64(25), np.int64(26), np.int64(48)]
Ticket 3 (Temp 0.7): [np.int64(5), np.int64(11), np.int64(21), np.int64(27), np.int64(31), np.int64(33)]


In [64]:
def check_winning_ticket(winning_numbers, generated_tickets):
    winning_set = set(winning_numbers)
    print("\n" + "="*40)
    print(f"  KIỂM TRA VÉ TRÚNG THƯỞNG")
    print(f"  Dãy số trúng thưởng: {sorted(winning_numbers)}")
    print("="*40)

    for ticket_info in generated_tickets:
        ticket_numbers = set(ticket_info['ticket'])
        common_numbers = winning_set.intersection(ticket_numbers)
        num_matches = len(common_numbers)

        print(f"Vé {ticket_info['id']} (Temp {ticket_info['temp']:.1f}): {ticket_info['ticket']}")
        print(f"  -> Số trùng khớp: {num_matches} ({sorted(list(common_numbers))})")

# Ví dụ sử dụng hàm check_winning_ticket
# Bạn có thể thay đổi dãy số trúng thưởng này thành dãy số thực tế.
winning_numbers_example = [14, 18, 21, 38, 48, 52] # Đây là kết quả kỳ quay 1402 trong dữ liệu

# Sử dụng biến final_lstm_tickets từ cell trước
# Đảm bảo cell trước (Mp3hlTK26ier) đã được chạy để biến này tồn tại
check_winning_ticket(winning_numbers_example, final_lstm_tickets)



  KIỂM TRA VÉ TRÚNG THƯỞNG
  Dãy số trúng thưởng: [14, 18, 21, 38, 48, 52]
Vé 1 (Temp 0.7): [np.int64(16), np.int64(19), np.int64(22), np.int64(24), np.int64(43), np.int64(45)]
  -> Số trùng khớp: 0 ([])
Vé 2 (Temp 0.7): [np.int64(3), np.int64(7), np.int64(17), np.int64(25), np.int64(26), np.int64(48)]
  -> Số trùng khớp: 1 ([np.int64(48)])
Vé 3 (Temp 0.7): [np.int64(5), np.int64(11), np.int64(21), np.int64(27), np.int64(31), np.int64(33)]
  -> Số trùng khớp: 1 ([np.int64(21)])


### Các chiến lược tối ưu hóa:

1.  **Tối ưu hóa Mô hình LSTM:**
    *   **Nhiều dữ liệu hơn:** Đảm bảo bạn đang sử dụng lượng dữ liệu lịch sử lớn nhất có thể.
    *   **Tinh chỉnh siêu tham số (Hyperparameter Tuning):** Các tham số như `seq_len` (độ dài chuỗi đầu vào), `hidden_size` (kích thước lớp ẩn), `learning_rate` (tốc độ học) và số `epochs` (số lần lặp huấn luyện) của mô hình LSTM có thể ảnh hưởng lớn đến hiệu suất. Bạn có thể sử dụng các kỹ thuật như tìm kiếm lưới (Grid Search) hoặc tìm kiếm ngẫu nhiên (Random Search) để tìm ra bộ siêu tham số tốt nhất.
    *   **Kiến trúc mô hình phức tạp hơn:** Thử nghiệm với các lớp LSTM sâu hơn (nhiều lớp `nn.LSTM` xếp chồng), hoặc sử dụng Bi-directional LSTM để mô hình có thể xem xét cả chuỗi dữ liệu xuôi và ngược.
    *   **Sử dụng kỹ thuật Regularization:** Thêm Dropout để tránh overfitting, đặc biệt nếu mô hình trở nên quá phức tạp.

2.  **Tối ưu hóa chiến lược lấy mẫu vé (`sample_ticket_from_probs`):**
    *   **Thử nghiệm Temperature có hệ thống:** Như bạn đã làm, thử các giá trị `temperature` khác nhau. Có thể tạo ra nhiều vé hơn với mỗi `temperature` và đánh giá xem vé nào có hiệu suất tốt nhất (dựa trên dữ liệu lịch sử hoặc mô phỏng).
    *   **Lấy mẫu Top-K hoặc Nucleus Sampling:** Thay vì chỉ dựa vào phân phối xác suất đơn thuần, các phương pháp này chọn các số từ một tập hợp các số có xác suất cao nhất, sau đó lấy mẫu từ tập hợp đó. Điều này có thể giúp tạo ra các vé đa dạng hơn nhưng vẫn có tính hợp lý.
    *   **Kết hợp các heuristic:** Ngoài xác suất, bạn có thể thêm các quy tắc (heuristics) để làm cho vé "thông minh hơn", ví dụ:
        *   Tránh các vé chỉ toàn số chẵn/lẻ.
        *   Tránh các vé có quá nhiều số liên tiếp.
        *   Cân bằng tổng các số trong vé.
        *   Tránh các vé đã trúng giải đặc biệt trong quá khứ (vì xác suất lặp lại rất thấp).

3.  **Đánh giá hiệu suất:**
    *   **Mô phỏng ngược (Backtesting):** Thay vì chỉ kiểm tra với một dãy số trúng thưởng, hãy chạy mô hình trên nhiều kỳ quay trong quá khứ và xem "tỷ lệ trúng" (số lượng số trùng khớp) của các vé được tạo ra. Điều này sẽ cho bạn cái nhìn khách quan hơn về hiệu quả của từng chiến lược hoặc tham số `temperature`.
    *   **Xác định chỉ số thành công:** Định nghĩa rõ "vé trúng nhiều số hơn" có nghĩa là gì. Là số lượng vé có 3 số trở lên? Hay là trung bình số lượng số trùng khớp trên mỗi vé?

### Bước tiếp theo đề xuất: Tinh chỉnh siêu tham số LSTM và đánh giá có hệ thống hơn

Tôi có thể giúp bạn bằng cách thêm mã để:

*   **Thử nghiệm các giá trị `seq_len`, `hidden_size`, `learning_rate` khác nhau cho LSTM.**
*   **Thực hiện một đánh giá tự động trên nhiều kỳ quay lịch sử** để xem chiến lược `temperature` nào cho kết quả "trùng khớp" tốt nhất một cách nhất quán.

Bạn muốn thử bắt đầu với bước nào trước?

In [8]:
# Helper function to train LSTM and get probabilities for a given historical dataset
def train_and_predict_lstm(historical_data, seq_len, total_numbers, epochs=10, lr=0.005):
    X_seq, y_seq = [], []
    # Ensure there's enough data for at least one sequence
    if len(historical_data) <= seq_len:
        # Return uniform probabilities if not enough data for a sequence
        return np.ones(total_numbers) / total_numbers

    for i in range(seq_len, len(historical_data)):
        seq_in = [np.eye(total_numbers)[np.array(draw) - 1].sum(axis=0) for draw in historical_data[i - seq_len:i]]
        seq_out = np.eye(total_numbers)[np.array(historical_data[i]) - 1].sum(axis=0)
        X_seq.append(seq_in)
        y_seq.append(seq_out)

    X_seq_t = torch.tensor(np.array(X_seq), dtype=torch.float32)
    y_seq_t = torch.tensor(np.array(y_seq), dtype=torch.float32)

    lstm = LSTMModel(input_size=total_numbers)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(lstm.parameters(), lr=lr)

    lstm.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = lstm(X_seq_t)
        loss = criterion(outputs, y_seq_t)
        loss.backward()
        optimizer.step()

    # Predict for the next draw based on the very last sequence in historical_data
    last_seq = [np.eye(total_numbers)[np.array(draw) - 1].sum(axis=0) for draw in historical_data[-seq_len:]]
    last_seq_t = torch.tensor(np.array([last_seq]), dtype=torch.float32)

    lstm.eval()
    with torch.no_grad():
        logits = lstm(last_seq_t)
        lstm_probs = torch.sigmoid(logits).numpy()[0]
    return lstm_probs


# =============================================================
# 3. BACKTESTING LSTM MODEL
# =============================================================
print("\n" + "="*40)
print("  BẮT ĐẦU BACKTESTING MÔ HÌNH LSTM")
print("="*40)

# Backtesting parameters
num_backtest_draws = 100 # Số kỳ quay gần nhất để backtest
start_backtest_idx = num_draws - num_backtest_draws

# Store results for each temperature
# Each temperature will have counts for 0, 1, ..., 6 matches
results_by_temp = {temp: {f'{i}_matches': 0 for i in range(7)} for temp in [0.5, 0.7, 0.8, 1.0, 1.3, 1.5, 2.0]}

print(f"Thực hiện backtesting trên {num_backtest_draws} kỳ quay gần nhất...")
print("Quá trình này có thể mất một khoảng thời gian do phải huấn luyện lại mô hình nhiều lần.\n")

for i in range(start_backtest_idx, num_draws):
    # current_historical_data contains data UP TO the draw BEFORE the one we are predicting
    current_historical_data = data[:i]
    winning_numbers = data[i] # This is the actual winning draw we are trying to predict

    # Re-train and predict LSTM for current historical data
    # Reducing epochs for faster backtesting, adjust as needed if more accuracy is desired
    current_lstm_probs = train_and_predict_lstm(current_historical_data, seq_len, total_numbers, epochs=10, lr=0.005)

    # Generate tickets for this draw using various temperatures
    generated_tickets_for_draw = []
    for temp_val in results_by_temp.keys(): # Use the same temperature values for consistency
        ticket = sample_ticket_from_probs(current_lstm_probs, num_tickets=1, temperature=temp_val)[0]
        generated_tickets_for_draw.append({
            'temp': temp_val,
            'ticket': ticket
        })

    # Check winning tickets for this draw and update overall results
    winning_set = set(winning_numbers)
    for ticket_info in generated_tickets_for_draw:
        ticket_numbers = set(ticket_info['ticket'])
        num_matches = len(winning_set.intersection(ticket_numbers))
        results_by_temp[ticket_info['temp']][f'{num_matches}_matches'] += 1

    if (i - start_backtest_idx + 1) % 10 == 0 or (i == num_draws - 1):
        print(f"Đã hoàn thành {i - start_backtest_idx + 1}/{num_backtest_draws} kỳ quay backtest.")

print("\n" + "="*40)
print("  KẾT QUẢ BACKTESTING TỔNG KẾT (LSTM)")
print("="*40)

print(f"\nSố lượng kỳ quay đã backtest: {num_backtest_draws}")
print("\nTổng quan kết quả trùng khớp theo Temperature:")
for temp, res in results_by_temp.items():
    print(f"  Temperature {temp:.1f}:")
    for matches_count_str, count in res.items():
        print(f"    - {matches_count_str}: {count} lần")
    print("-" * 20)


  BẮT ĐẦU BACKTESTING MÔ HÌNH LSTM
Thực hiện backtesting trên 100 kỳ quay gần nhất...
Quá trình này có thể mất một khoảng thời gian do phải huấn luyện lại mô hình nhiều lần.

Đã hoàn thành 10/100 kỳ quay backtest.
Đã hoàn thành 20/100 kỳ quay backtest.
Đã hoàn thành 30/100 kỳ quay backtest.
Đã hoàn thành 40/100 kỳ quay backtest.
Đã hoàn thành 50/100 kỳ quay backtest.
Đã hoàn thành 60/100 kỳ quay backtest.
Đã hoàn thành 70/100 kỳ quay backtest.
Đã hoàn thành 80/100 kỳ quay backtest.
Đã hoàn thành 90/100 kỳ quay backtest.
Đã hoàn thành 100/100 kỳ quay backtest.

  KẾT QUẢ BACKTESTING TỔNG KẾT (LSTM)

Số lượng kỳ quay đã backtest: 100

Tổng quan kết quả trùng khớp theo Temperature:
  Temperature 0.5:
    - 0_matches: 46 lần
    - 1_matches: 42 lần
    - 2_matches: 10 lần
    - 3_matches: 2 lần
    - 4_matches: 0 lần
    - 5_matches: 0 lần
    - 6_matches: 0 lần
--------------------
  Temperature 0.7:
    - 0_matches: 42 lần
    - 1_matches: 45 lần
    - 2_matches: 12 lần
    - 3_matches: